In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import requests
import time
import json
import urllib.request
import urllib.parse
import socket
from astropy.io import fits
from astropy.wcs import WCS

## Image Processing and Star Detection

The input image is loaded in grayscale and processed using adaptive thresholding to isolate bright spots (stars) from the background. Connected component analysis is then applied to find and filter star blobs by area, removing small noise artifacts. The resulting centroids represent the pixel coordinates of detected stars in the image.

In [ ]:
# Manually specify the image path
img_path = Images/Image1.jpg # Your input image path here

print(f"Image path set to: {img_path}")

# Load the image in grayscale
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    print(f"✗ Failed to load image. Please update img_path with a valid file path")
else:
    print(f"✓ Image loaded successfully: {img.shape}")

In [ ]:
plt.imshow(img, cmap='gray')
plt.title("Input Image")

In [37]:
# Thresholding to isolate bright spots
thresh = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 17, 2
)

# Find connected components (stars)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)

# Remove background and noise
star_centroids = []
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if area > 5:  # filter tiny noise blobs
        star_centroids.append(centroids[i])

# Convert to NumPy array
star_centroids = np.array(star_centroids)

In [ ]:
# Show detected stars on the original image
output = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
for (x, y) in star_centroids:
    cv2.circle(output, (int(x), int(y)), 4, (0, 0, 255), 1)

plt.figure(figsize=(8, 8))
plt.imshow(output[..., ::-1])  # Convert BGR to RGB for matplotlib
plt.title(f"Detected Stars: {len(star_centroids)}")
plt.show()

## Database Comparison

Detected star centroids are submitted to the [Astrometry.net](https://nova.astrometry.net) API, which plate-solves the image by matching the detected star pattern against a large catalog of known star positions. A session key is required for authentication — set your API key before running this section. Once solved, a WCS (World Coordinate System) file is downloaded containing the coordinate transformation data needed to map pixel positions to sky coordinates.

In [ ]:
API_URL = "http://nova.astrometry.net/api/"
API_KEY = "your_api_key_here"   # Your Astrometry.net API key

# --- Login with proper headers ---
headers = {"Content-Type": "application/x-www-form-urlencoded"}
payload = f"request-json={json.dumps({'apikey': API_KEY})}"

r = requests.post(API_URL + "login", data=payload, headers=headers)
resp = r.json()

if resp.get("status") == "success":
    session = resp["session"]
    print(f"✓ Successfully logged in")
    print(f"  User: {resp.get('message')}")
    print(f"  Session: {session}")
else:
    print(f"✗ Login failed: {resp.get('errormessage', resp)}")
    session = None

In [ ]:
# --- Upload the image ---

if session is None:
    print("Error: Not logged in. Run the login cell first.")
else:
    image_path = img_path
    
    # Upload with proper API format.
    # The request-json field must be part of the multipart form, not a separate data param.
    with open(image_path, "rb") as f:
        files = {
            "file": f,
            "request-json": (None, json.dumps({
                'session': session, 
                'allow_commercial_use': 'y', 
                'allow_modifications': 'y'
            }))
        }
        r = requests.post(API_URL + "upload", files=files)
    
    resp = r.json()
    print(f"Upload response status: {resp.get('status')}")
    
    if resp.get("status") == "success":
        subid = resp["subid"]
        print(f"✓ Submitted as job: {subid}")

        # --- Wait for job assignment ---
        job_id = None
        for attempt in range(60):  # Try for up to 5 minutes
            try:
                params = {"request-json": json.dumps({'session': session})}
                r = requests.get(API_URL + f"submissions/{subid}", params=params, timeout=10)
                resp_sub = r.json()
                jobs = resp_sub.get("jobs", [])
                if jobs and jobs[0] is not None:
                    job_id = jobs[0]
                    print(f"✓ Job ID: {job_id}")
                    break
                # Debug: show what we're getting back
                if attempt == 0 or (attempt + 1) % 20 == 0:
                    print(f"  Submission status: {resp_sub.get('processing_started', 'not started')}, jobs: {jobs}")
            except Exception as e:
                print(f"  Warning: {type(e).__name__}, retrying...")
            
            print(f"  Waiting for job assignment... (attempt {attempt+1}/60)")
            time.sleep(5)

        if job_id:
            # --- Wait for solve results ---
            print("  Waiting for solution...")
            for attempt in range(300):  # Try for up to 25 minutes
                try:
                    r = requests.get(API_URL + f"jobs/{job_id}", timeout=10)
                    resp_job = r.json()
                    status = resp_job.get("status")
                    if status == "success":
                        print("✓ Solved!")
                        break
                    elif status == "failure":
                        print(f"✗ Solve failed: {resp_job.get('errormessage', 'unknown')}")
                        break
                except Exception as e:
                    print(f"  Warning: {type(e).__name__}, retrying...")
                    continue
                
                if (attempt + 1) % 12 == 0:  # Print every minute
                    print(f"  Status: {status} (attempt {attempt+1}/300)")
                time.sleep(5)

            # --- Get WCS and FITS results ---
            # The API documentation shows two separate URLs for these files:
            #   http://nova.astrometry.net/wcs_file/JOBID
            #   http://nova.astrometry.net/new_fits_file/JOBID
            # and they require a Referer header when downloaded programmatically.
            try:
                params = {"request-json": json.dumps({'session': session})}
                referer = "https://nova.astrometry.net/api/login"
                headers_download = {"Referer": referer}

                # fetch WCS text
                r = requests.get(
                    f"http://nova.astrometry.net/wcs_file/{job_id}",
                    params=params,
                    timeout=20,
                    headers=headers_download,
                )
                ct = r.headers.get("Content-Type", "")
                print("WCS content-type:", ct)
                # the server sometimes uses 'application/fits' for the plain-text
                # WCS file; the important thing is to reject HTML error pages.
                if r.status_code == 200 and "html" not in ct.lower():
                    open("solution.wcs", "wb").write(r.content)
                    print("✓ WCS saved")
                else:
                    print(f"✗ Failed to get WCS ({r.status_code})")
                    print(r.text[:200])

                # Note: The server rate-limits full FITS downloads (~72MB) at ~6.4MB.
                # The WCS file contains all the coordinate transformation info we need,
                # so we skip the FITS download and use only the WCS.
            except Exception as e:
                print(f"✗ Error getting WCS/FITS: {e}")
        else:
            print("✗ Could not get job ID after 5 minutes")
    else:
        print(f"✗ Upload failed: {resp}")

## Processing WCS File

The WCS endpoint requires a `Referer` header or you'll get an HTML error page
back. The WCS file comes from `/wcs_file/JOBID` (not under `/api/`).

The server rate-limits the full FITS file download (`/new_fits_file/JOBID`)
at around 6.4MB out of ~72MB, making it unreliable. Since the WCS file
contains all the coordinate transformation information we need, we download
only the WCS file and skip the FITS download.

In [ ]:
# Read WCS directly from the text file, extracting only valid WCS cards
with open("solution.wcs") as f:
    wcs_text = f.read()

# Parse the WCS file line by line, skipping problematic CONTINUE cards
lines = wcs_text.strip().split('\n')
valid_lines = []
for line in lines:
    # Skip empty lines and CONTINUE cards (which are malformed in this file)
    if line.strip() and not line.startswith('CONTINUE'):
        valid_lines.append(line)

# Reconstruct the header as plain text (don't create Header object)
clean_wcs_text = '\n'.join(valid_lines)

# Pass string directly to WCS to avoid Header parsing issues
w = WCS(clean_wcs_text)

print("WCS type:", w.wcs.ctype)
print("Has celestial:", w.has_celestial)

In [ ]:
# Transform detected star pixel coordinates to celestial coordinates using WCS
xy = star_centroids  # Nx2 array of pixel coordinates
ra, dec = w.all_pix2world(xy[:,0], xy[:,1], 0)

print("Star coordinates (RA, Dec):")
print(np.column_stack([ra, dec]))
print(f"\nSolved {len(ra)} stars")
print(f"Sky center: RA={np.mean(ra):.2f}°, Dec={np.mean(dec):.2f}°")

## Vector Calculation and Output

Using the WCS solution, each detected star's pixel coordinates are converted to celestial coordinates (Right Ascension and Declination). The image center is then mapped to the sky, and a small northward step in Declination is projected back into pixel space to compute the direction of Celestial North. This north-pointing arrow is drawn onto the original image and displayed as the final output.

In [ ]:
# Processing Image

# Load image
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
if img is None:
    print(f"✗ Failed to load image from {img_path}")
else:
    # Detect stars
    thresh = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 17, 2
    )
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh)
    
    star_centroids = []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area > 5:
            star_centroids.append(centroids[i])
    
    star_centroids = np.array(star_centroids)
    print(f"✓ Detected {len(star_centroids)} stars")
    
    # Use the WCS object from previous cell
    wcs_obj = w
    
    # Get image center in pixels
    img_height, img_width = img.shape[:2]
    x_center = img_width / 2
    y_center = img_height / 2

    # Convert center pixel -> RA/Dec
    ra_c, dec_c = wcs_obj.all_pix2world(x_center, y_center, 0)

    # Move slightly toward celestial north
    dec_step = dec_c + 0.1   # 0.1 degree step toward pole
    ra_step  = ra_c          # keep RA constant

    # Convert new sky coord back -> pixel
    x_north, y_north = wcs_obj.all_world2pix(ra_step, dec_step, 0)

    # Check if the north point is within image bounds
    if not (0 <= x_north < img_width and 0 <= y_north < img_height):
        dec_step = dec_c + 0.05
        x_north, y_north = wcs_obj.all_world2pix(ra_step, dec_step, 0)

    # Compute direction vector in image plane
    dx = x_north - x_center
    dy = y_north - y_center

    vec = np.array([dx, dy])
    vec_magnitude = np.linalg.norm(vec)

    if vec_magnitude < 1e-6:
        print(f"⚠ Warning: North direction vector is too small")
    else:
        vec = vec / vec_magnitude   # normalize

        # Scale for drawing
        arrow_length = max(100, min(400, vec_magnitude * 20))
        dx_draw = vec[0] * arrow_length
        dy_draw = vec[1] * arrow_length

        # Prepare image for drawing
        if len(img.shape) == 2:
            image_with_arrow = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        else:
            image_with_arrow = img.copy()

        # Draw arrow on image
        start = (int(x_center), int(y_center))
        end   = (int(x_center + dx_draw), int(y_center + dy_draw))

        cv2.arrowedLine(
            image_with_arrow,
            start,
            end,
            (50, 205, 50),   # Neon Green in BGR
            20,            # THICK LINE
            tipLength=0.5
        )

        # Display with matplotlib
        plt.figure(figsize=(12, 8))
        plt.imshow(image_with_arrow[..., ::-1])
        plt.title(f"Celestial North Direction")
        plt.axis('off')
        plt.text(10, 30, 'Arrow points to North', color='white', fontsize=12, 
                 bbox=dict(facecolor='black', alpha=0.5))
        plt.tight_layout()
        plt.show()

        print(f"✓ Image processed successfully")
        print(f"  Arrow length: {arrow_length:.1f} px, Direction: ({vec[0]:.2f}, {vec[1]:.2f})")